# Homework — Plotting with matplotlib

A plot is an *argument*. A good one makes a single idea obvious at a glance. In this
homework you'll build the four plots you'll use for the rest of your life, and — just as
important — practice **deciding which plot the question actually needs**.

**How this works**
1. Run the setup cell below.
2. Read each short teaching section and run its example (a plot will appear).
3. Answer the numbered **Problems** in the empty cells. About half ask you to *think*,
   not just to plot — write those answers as a `# comment` or a `print(...)`.
4. At the very end there's a **check-in value** — type it into the CHEM 438 app to
   mark this homework done.

Run every cell top to bottom. If you skip the setup cell, nothing else will work.

<!-- c438-copy-note -->
> ### ▶️ First — make your own copy
> This notebook opened **read-only** (it comes straight from GitHub). To type your answers and keep your work:
>
> **File ▸ Save a copy in Drive**  —  or click **Copy to Drive** in the toolbar.
>
> Then work in *your* copy. It saves to your Google Drive automatically.


In [ ]:
# Setup — run this first
!pip install matplotlib -q
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
print("Ready to plot.")

## First: which plot?

Picking the plot is the real skill. Match the **question** to the plot:

| You want to show...                                   | Use a...      | matplotlib call |
|-------------------------------------------------------|---------------|-----------------|
| how something **changes over an ordered axis** (time) | **line**      | `plt.plot`      |
| the **relationship between two measured numbers**     | **scatter**   | `plt.scatter`   |
| a **value for each category** (compare groups)        | **bar**       | `plt.bar`       |
| the **shape / spread of one column of numbers**       | **histogram** | `plt.hist`      |

Rules of thumb:
- **Line** connects its points, so it implies the x-axis has a meaningful order
  (Jan -> Feb -> Mar). Never connect unordered categories with a line — it invents a trend
  that isn't there.
- **Scatter** does *not* connect points, so it's honest about two independent measurements.
- **Bar** is for a handful of labeled categories.
- **Histogram** throws away the individual labels and shows the *distribution* — where the
  numbers pile up, and whether the tail leans left or right.

**Quick self-check (answer in your head):** You have the boiling points of 5 different
solvents and want to compare them. Which plot?
&nbsp;&nbsp;(a) line &nbsp; (b) scatter &nbsp; (c) bar &nbsp; (d) histogram

> Answer: **(c) bar**. Five named categories, one value each. A line would falsely suggest
> the solvents are steps in a sequence; a histogram would hide their names.

## 1. Line plot — a trend over an ordered axis

Below are the measured yields of a reaction run once a month for a year. Month is ordered,
so a **line** is right: it shows the trend rising and dipping through the year.

Notice every plot in this notebook has a **title**, an **x-label**, and a **y-label**. A
plot with no axis labels is useless — the reader has no idea what they're looking at.
**Label everything.**

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
yield_pct = [72, 68, 81, 75, 90, 88, 79, 95, 84, 77, 82, 91]
df = pd.DataFrame({"Month": months, "Yield": yield_pct})

plt.plot(df["Month"], df["Yield"], marker="o", color="teal")   # plot straight from a column
plt.title("Reaction yield by month")
plt.xlabel("Month")
plt.ylabel("Yield (%)")
plt.show()

# A plot prints no number, so pull a real one out of the data:
print("Average yield:", round(df["Yield"].mean(), 1), "%")

## 2. Scatter plot — a relationship between two numbers

We measured reaction rate at eight temperatures. The question is "does rate depend on
temperature?" — a relationship between two measured numbers — so we use **scatter**. The
points are *not* connected, because each measurement stands on its own.

In [ ]:
temp = [10, 20, 30, 40, 50, 60, 70, 80]              # deg C
rate = [1.2, 2.1, 3.9, 5.0, 8.2, 11.5, 15.1, 20.0]   # mM/s

plt.scatter(temp, rate, color="darkorange")
plt.title("Reaction rate vs temperature")
plt.xlabel("Temperature (deg C)")
plt.ylabel("Rate (mM/s)")
plt.show()

# The cloud of points clearly trends upward. Correlation puts a number on it:
print("Correlation:", round(np.corrcoef(temp, rate)[0, 1], 2))

## 3. Bar chart — a value per category

Five solvents, one measured boiling point each. These are **categories**, not a sequence,
so we use **bar**. (Use `plt.barh` for a *horizontal* bar chart, which reads better when the
category names are long.)

In [ ]:
solvent = ["Water", "Ethanol", "Acetone", "Hexane", "Toluene"]
bp = [100, 78, 56, 69, 111]   # boiling point, deg C

plt.bar(solvent, bp, color="steelblue")
plt.title("Boiling point by solvent")
plt.xlabel("Solvent")
plt.ylabel("Boiling point (deg C)")
plt.show()

# Which solvent boils highest?
print("Highest boiling:", solvent[int(np.argmax(bp))])

## 4. Histogram — the shape of one column of numbers

A histogram takes **one column of numbers**, chops the range into bins, and counts how many
values land in each bin. It answers "what's the spread?" and "is it lopsided?" — *not*
"what's the value for X?". Here are 40 repeated pH measurements of the same buffer.

In [ ]:
rng = np.random.default_rng(0)
ph = np.round(rng.normal(7.4, 0.15, 40), 2)   # 40 measurements clustered near 7.4

plt.hist(ph, bins=8, color="mediumpurple", edgecolor="black")
plt.title("Distribution of pH measurements")
plt.xlabel("pH")
plt.ylabel("Count")   # the y-axis of a histogram is always a COUNT
plt.show()

print("Measurements pile up near the mean:", round(ph.mean(), 2))

## 5. Plotting a summary — where pandas meets matplotlib

This is the one you will actually use. Data almost never arrives ready to plot: it arrives
as a table with one row per measurement, and the figure you want is one bar per *group*.

**Group, summarise, reset, plot.** Four steps, and you know all of them.


In [ ]:
runs = pd.DataFrame({
    "run":      ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8", "R9"],
    "solvent":  ["water", "water", "water", "ethanol", "ethanol", "ethanol",
                 "hexane", "hexane", "hexane"],
    "yield_pct":[42, 68, 61, 55, 84, 79, 30, 47, 38],
})

# 1. group  2. summarise  3. reset  -> an ordinary two-column table
summary = runs.groupby("solvent")["yield_pct"].mean().round(1).reset_index()
print(summary)
#    solvent  yield_pct
# 0  ethanol       72.7
# 1   hexane       38.3
# 2    water       57.0

# 4. plot it — the two columns are exactly what plt.bar wants
plt.bar(summary["solvent"], summary["yield_pct"], color="seagreen", edgecolor="black")
plt.xlabel("Solvent")
plt.ylabel("Mean yield (%)")
plt.title("Mean yield by solvent")
plt.show()


Without `reset_index()` the solvent names are the **index**, not a column, and
`summary["solvent"]` raises `KeyError`. That is the whole reason it is there.


**Problem 8 (put it together).** Using the same `runs` table:

1. Build a summary of the **highest** yield per solvent, with `reset_index()`.
2. Make a bar chart of it, labelled properly — axis labels and a title.
3. In a comment, say which solvent you would run the reaction in, and why.


## Labels are not optional (and axes can lie)

Two habits that separate a real figure from a bad one:

1. **Always label axes and give a title.**
2. **Watch where the y-axis starts.**

> **A note on `plt.subplots` below.** The next cell puts two plots side by side using
> `fig, (ax_bad, ax_good) = plt.subplots(...)`, which is Tuesday's lecture. Read it as a
> demonstration — nothing you are asked to do needs it. `ax.set_title` is just the
> `plt.title` you already know, spoken to one panel instead of the whole figure.


In [ ]:
names = ["A", "B", "C", "D"]
score = [96, 97, 95, 98]

fig, (ax_bad, ax_good) = plt.subplots(1, 2, figsize=(10, 4))

# BAD: no labels, no title, y-axis starts at 90 -> differences look huge
ax_bad.bar(names, score, color="crimson")
ax_bad.set_ylim(90, 100)

# GOOD: labelled, honest y-axis from 0
ax_good.bar(names, score, color="seagreen")
ax_good.set_ylim(0, 100)
ax_good.set_title("Assay score by sample")
ax_good.set_xlabel("Sample")
ax_good.set_ylabel("Score")

plt.show()
print("Real spread:", max(score) - min(score), "points (tiny!)")

## One seaborn example (a preview of Tuesday)

**seaborn** is a thin layer on top of matplotlib for quick statistical plots. Here is one
example so you have seen it — Tuesday's lecture is all seaborn. Nothing in the problems
below needs it.


In [ ]:
!pip install seaborn -q
import seaborn as sns

# Same idea as our histogram, but seaborn adds a smooth curve and styling in one line.
rng = np.random.default_rng(0)
ph = rng.normal(7.4, 0.15, 40)

sns.histplot(ph, bins=8, kde=True, color="teal")
plt.title("pH distribution (seaborn)")
plt.xlabel("pH")
plt.ylabel("Count")
plt.show()

## Problems

Write your code (or your written answer) in the empty cell under each problem, then run it.
The **reasoning** problems have no plot to make — answer in a `# comment` or a
`print("...")`. There's no single right wording; we're after your reasoning.

**Problem 1 (reasoning — choose the plot).** You measured a beaker's temperature every 10
minutes for one hour as it cooled, and you want to show how it changed over that hour. Which
plot — line, scatter, bar, or histogram? Write your choice and **one sentence** saying
why.

**Problem 2 (make a plot).** Two reactions were run side by side for six months. Plot
**both** yield series on one line chart so they can be compared. Requirements: markers, a
title, x- and y-axis labels, and a **legend** (so the reader knows which line is which).

```python
data = {
    "Month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "Reaction A": [120, 135, 150, 140, 160, 175],
    "Reaction B": [100, 115, 130, 125, 145, 155],
}
```
Hint: call `plt.plot(...)` twice with a `label=` on each, then `plt.legend()`.

**Problem 3 (reasoning — read a plot).** You make a histogram of how long a reaction takes
to finish across 100 runs. Most runs pile up on the left (fast), with a long tail stretching
to the right (a few very slow runs). This shape is called **skewed right**. In a sentence or
two: what does that tell you about the reaction times? And which is pulled higher by the
tail — the **mean** or the **median**?

**Problem 4 (make a plot).** For a calibration curve you measured absorbance at several
known concentrations. Make a **scatter** plot of absorbance (y) vs concentration (x) with a
title and both axis labels. (Why scatter, not line? You're showing a relationship between
two measured numbers.)

```python
concentration = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]   # mM
absorbance    = [0.09, 0.21, 0.38, 0.63, 0.79, 1.01]
```

**Problem 5 (reasoning — critique).** A classmate shows you a bar chart with these
problems: **no axis labels**, **no title**, and a **y-axis that starts at 95** (not 0) so
four nearly identical bars look wildly different. Name **two** things wrong with the chart,
and for the y-axis issue say what it misleads the reader into believing.

**Problem 6 (reasoning — predict).** You're about to scatter-plot reaction rates. Almost
every value is between 1 and 20, but one bad run recorded **500**. Before plotting, predict
what the scatter will look like with that outlier included: what happens to the y-axis, and
what happens to all the normal points? Name one thing you could do about it.

**Problem 7 (reasoning — compare).** The five-solvent boiling-point data from the teaching
section could be drawn as a **bar** chart or as a **line** chart. Which one reads better for
this data, and why is the other misleading? (Hint: think about what *connecting the points
with a line* implies about the solvents.)

## Check in

The teaching section plotted **12 months of reaction yields**. The cell below computes one
number from that exact data: **how many months had a yield *above* the average yield?**

Run it and **type that number into the CHEM 438 app** to mark this homework complete.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
yield_pct = [72, 68, 81, 75, 90, 88, 79, 95, 84, 77, 82, 91]
df = pd.DataFrame({"Month": months, "Yield": yield_pct})

above_average = int((df["Yield"] > df["Yield"].mean()).sum())
print("Your check-in value:", above_average)